# 03 - Analisis univariado

**Responsable principal:** Daniela Ramirez de Leon

**Actividad de la guia:** 4b (resumen de variables numericas y tablas de frecuencia para categoricas) y parte de *Analisis Exploratorio* (30 pts).

**Objetivo de este notebook:** estadistica descriptiva variable por variable. Independiente de 01/02/04/05 (usa `train.csv` y la muestra fija de landmarks de `config.SAMPLE_LANDMARK_PATHS`).

In [3]:
import sys
sys.path.append("..")

from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src import data_loading as dl

sns.set_theme(style="whitegrid")


## 1. Variables numericas: estadistica descriptiva

`train.csv` no trae variables numericas que valga la pena describir por si solas: `file_id`, `sequence_id` y `participant_id` son identificadores. El analisis univariado se hace entonces sobre tres variables derivadas:

- `phrase_len`: numero de caracteres de la frase objetivo (todas las secuencias de `train.csv`).
- `seq_len`: numero de frames de la secuencia (muestra fija de landmarks).
- `pct_sin_mano`: porcentaje de frames de la secuencia sin ninguna mano detectada (muestra fija).

In [6]:
train_df = dl.load_train_index()
train_df["phrase_len"] = train_df["phrase"].str.len()

print("Secuencias:", len(train_df))
train_df.dtypes


Secuencias: 67208


path              object
file_id            int64
sequence_id        int64
participant_id     int64
phrase            object
phrase_len         int64
dtype: object

In [7]:
# Solo se leen las columnas x de las manos, para no cargar las 1,630 columnas del parquet.
COLS_MANOS = [f"x_{mano}_{i}" for mano in ("left_hand", "right_hand") for i in range(21)]

partes = []
for rel_path in config.SAMPLE_LANDMARK_PATHS:
    lm = pd.read_parquet(dl.landmark_path(rel_path), columns=COLS_MANOS)
    mano_visible = lm.notna().any(axis=1)
    partes.append(pd.DataFrame({
        "seq_len": lm.groupby(lm.index).size(),
        "pct_sin_mano": 100 * (1 - mano_visible.groupby(lm.index).mean()),
    }))

seq_df = pd.concat(partes).reset_index()
print("Secuencias en la muestra:", len(seq_df))
seq_df.head()


Secuencias en la muestra: 1997


,sequence_id,seq_len,pct_sin_mano
0,1975433633,127,83.464567
1,1975473601,272,4.411765
2,1975502450,253,3.162055
3,1975521182,126,81.746032
4,1975541698,150,14.000000


In [8]:
def resumen_numerico(df, columnas):
    """describe() mas rango intercuartil, coeficiente de variacion y asimetria."""
    resumen = df[columnas].describe().T
    resumen["IQR"] = resumen["75%"] - resumen["25%"]
    resumen["CV"] = resumen["std"] / resumen["mean"]
    resumen["asimetria"] = df[columnas].skew()
    return resumen.round(2)


resumen_numerico(train_df, ["phrase_len"])


,count,mean,std,min,25%,50%,75%,max,IQR,CV,asimetria
phrase_len,67208.0,17.8,5.73,1.0,12.0,17.0,22.0,31.0,10.0,0.32,0.42


Las frases van de 1 a 31 caracteres, con media 17.8 y mediana 17: la distribucion es casi simetrica y esta acotada por arriba. Ese tope de 31 parece un limite de diseno en la generacion de las frases (direcciones, telefonos, URLs) y no algo propio del lenguaje. La dispersion es baja frente a la media (CV cercano a 0.32), asi que casi todas las frases caen en un rango estrecho.

In [9]:
resumen_numerico(seq_df, ["seq_len", "pct_sin_mano"])


,count,mean,std,min,25%,50%,75%,max,IQR,CV,asimetria
seq_len,1997.0,160.81,86.49,3.0,106.00,147.00,206.00,751.00,100.0,0.54,1.09
pct_sin_mano,1997.0,44.82,29.41,0.0,18.97,42.86,70.56,99.06,51.6,0.66,0.10


La longitud de secuencia se comporta distinto: la mediana ronda los 147 frames pero la cola llega arriba de 700, la media queda por encima de la mediana y la asimetria es positiva. Dicho de otra forma, frases de largo parecido pueden tomar duraciones muy distintas.

`pct_sin_mano` es la variable de calidad de los datos: mide que porcentaje de la secuencia no aporta senal de deletreo. No esta repartida de forma pareja entre secuencias, hay algunas donde la mano se pierde durante buena parte del video (es el mismo problema que se trato en el notebook 02).

## 2. Variables categoricas: tablas de frecuencia

Las dos variables categoricas del dataset son el participante que ejecuta el deletreo y los caracteres que componen las frases objetivo.

In [10]:
freq_part = train_df["participant_id"].value_counts().to_frame("secuencias")
freq_part["proporcion"] = freq_part["secuencias"] / len(train_df)

print("Participantes:", len(freq_part))
print("Secuencias por participante -> min:", freq_part["secuencias"].min(),
      "| mediana:", freq_part["secuencias"].median(),
      "| max:", freq_part["secuencias"].max())
print(f"Los 10 con mas datos concentran {100 * freq_part['proporcion'].head(10).sum():.1f}% del total")

freq_part.head(10).round(4)


Participantes: 94
Secuencias por participante -> min: 1 | mediana: 794.5 | max: 1535
Los 10 con mas datos concentran 14.5% del total


,secuencias,proporcion
participant_id,,
36,1535,0.0228
105,1006,0.0150
112,953,0.0142
81,944,0.0140
89,896,0.0133
188,895,0.0133
56,890,0.0132
178,886,0.0132
141,886,0.0132


Hay 94 participantes y el reparto es muy desigual: el que menos aporta tiene 1 sola secuencia y el que mas 1,535, con mediana de 794. Ninguno pasa del 2.5% del total, pero la diferencia entre extremos es de tres ordenes de magnitud, asi que los participantes con pocas secuencias practicamente no estan representados.

In [11]:
char_freq = pd.Series(Counter("".join(train_df["phrase"].dropna()))).sort_values(ascending=False).to_frame("frecuencia")
char_freq["proporcion"] = char_freq["frecuencia"] / char_freq["frecuencia"].sum()

print("Caracteres distintos:", len(char_freq))
char_freq.head(15).round(4)


Caracteres distintos: 59


,frecuencia,proporcion
e,71986,0.0602
a,69521,0.0581
,58569,0.0489
o,57784,0.0483
r,55533,0.0464
-,54242,0.0453
t,47132,0.0394
n,45663,0.0382
i,43840,0.0366
s,43224,0.0361


In [12]:
char_freq.tail(10).round(5)


,frecuencia,proporcion
~,21,0.00002
(,16,0.00001
),16,0.00001
$,5,0.00000
',3,0.00000
#,3,0.00000
!,3,0.00000
*,3,0.00000
[,2,0.00000
;,1,0.00000


In [13]:
def tipo_caracter(c):
    if c.isalpha():
        return "letra"
    if c.isdigit():
        return "digito"
    if c == " ":
        return "espacio"
    return "simbolo"


char_freq["tipo"] = [tipo_caracter(c) for c in char_freq.index]
resumen_tipos = char_freq.groupby("tipo")["frecuencia"].agg(distintos="count", total="sum")
resumen_tipos["proporcion"] = resumen_tipos["total"] / resumen_tipos["total"].sum()
resumen_tipos.round(4)


,distintos,total,proporcion
tipo,,,
digito,10,294261,0.2459
espacio,1,58569,0.0489
letra,26,732568,0.6122
simbolo,22,111216,0.0929


El alfabeto es pequeno pero muy desbalanceado. Arriba estan el espacio y las vocales e, a, o junto con r, n y t; abajo quedan letras como j, q y z y varios simbolos que aparecen en una fraccion minima de los casos. Los digitos pesan bastante porque muchas frases son direcciones y numeros de telefono, no texto corrido.

## 3. Observaciones

Las variables utiles no vienen dadas. `train.csv` solo trae identificadores y la frase objetivo, asi que el analisis univariado se sostiene en tres variables derivadas: longitud de frase, longitud de secuencia y porcentaje de frames sin mano detectada.

Longitud de frase y longitud de secuencia se comportan de manera distinta. La frase esta acotada y es casi simetrica (1 a 31 caracteres, media 17.8), mientras que la secuencia es asimetrica a la derecha, con mediana cercana a 147 frames y valores extremos que pasan de 700. Un mismo contenido puede tomar duraciones muy distintas, asi que la entrada del modelo tendra longitudes variables que habra que recortar o rellenar.

El desbalance es el patron que se repite en las dos variables categoricas. Entre participantes, de 1 a 1,535 secuencias por persona; entre caracteres, unas pocas letras concentran la mayoria de las apariciones mientras que j, q, z y los simbolos casi no tienen muestras. Las dos cosas afectan la evaluacion: conviene separar entrenamiento y validacion por participante, para no terminar midiendo si el modelo memorizo el estilo de deletreo de unos pocos, y revisar el desempeno por caracter y no solo el promedio general.

El porcentaje de frames sin mano detectada no es un detalle tecnico sino la variable de calidad del dataset: cuando la mano no aparece, el frame no aporta nada al deletreo aunque el resto de landmarks este completo. Esto respalda lo decidido en el notebook 02 (forward-fill dentro de cada secuencia y descarte de las secuencias sin mano util) y conviene reportarlo junto con cualquier resultado posterior.